# Решения: kNN и масштаб

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import pandas as pd


DATA_URL = (
    "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/modules/08_04_mnist_knn/data/digits.csv"
)


def find_digits_csv():
    for p in (
        Path("digits.csv"),
        Path("../digits.csv"),
        Path("../../data/digits.csv"),
        Path("../data/digits.csv"),
        Path("../../../data/digits.csv"),
    ):
        if p.exists():
            return p.resolve()
    return DATA_URL


DIGITS_PATH = find_digits_csv()
df = pd.read_csv(DIGITS_PATH)
PIXELS = [c for c in df.columns if c.startswith('p')]

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

train_part = df.iloc[:1200].reset_index(drop=True)
test_part = df.iloc[1200:].reset_index(drop=True)
train_pixels = train_part[PIXELS].to_numpy().tolist()
train_labels = train_part['label'].tolist()


## Урок. 1–3. Расстояние, сосед, голосование

In [ ]:
def distance(a, b):
    return sum((x - y) ** 2 for x, y in zip(a, b)) ** 0.5


probe = test_part.loc[0, PIXELS].tolist()
dists = [distance(probe, row) for row in train_pixels]
nn_index = min(range(len(dists)), key=lambda j: dists[j])
nn_label = train_labels[nn_index]
nn_dist = dists[nn_index]
order = sorted(range(len(dists)), key=lambda j: dists[j])[:3]
three_labels = [train_labels[j] for j in order]
vote_label = int(pd.Series(three_labels).value_counts().idxmax())
print(nn_index, nn_label, round(nn_dist, 2), three_labels, vote_label,
      test_part.loc[0, 'label'])

## Урок. 4–5. Масштаб и min-max

In [ ]:
row_a = df.loc[0, PIXELS].tolist()
row_b = df.loc[1, PIXELS].tolist()
ink_a, ink_b = sum(row_a) * 1000, sum(row_b) * 1000
pix_part = sum((x - y) ** 2 for x, y in zip(row_a, row_b))
ink_part = (ink_a - ink_b) ** 2
share_from_ink = ink_part / (pix_part + ink_part)


def scale_min_max(frame):
    rng = (frame.max() - frame.min()).replace(0, 1)
    return (frame - frame.min()) / rng


scaled_pixels = scale_min_max(df[PIXELS])
print(round(share_from_ink, 4), int(scaled_pixels.isna().sum().sum()))

## Урок. 6–8. sklearn kNN, выбор k, сверка

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(df[PIXELS], df['label'], test_size=0.25,
                                          random_state=0, stratify=df['label'])
model = KNeighborsClassifier(n_neighbors=3).fit(X_tr, y_tr)
acc_knn = float(accuracy_score(y_te, model.predict(X_te)))
acc_1 = float(accuracy_score(y_te, KNeighborsClassifier(1).fit(X_tr, y_tr).predict(X_te)))
acc_25 = float(accuracy_score(y_te, KNeighborsClassifier(25).fit(X_tr, y_tr).predict(X_te)))
K_NOTE = (
    'При k=25 в голосование попадают далёкие картинки других цифр, '
    'ответ усредняется и редкие начертания теряются.'
)
my_preds = []
for i in range(50):
    p = test_part.loc[i, PIXELS].tolist()
    d = [distance(p, row) for row in train_pixels]
    my_preds.append(train_labels[min(range(len(d)), key=lambda j: d[j])])
lib_preds = KNeighborsClassifier(1).fit(train_part[PIXELS], train_part['label'])\
    .predict(test_part[PIXELS].head(50)).tolist()
agree_share = sum(1 for a, b in zip(my_preds, lib_preds) if a == b) / 50
print(round(acc_knn, 4), round(acc_1, 4), round(acc_25, 4), agree_share)

## ДЗ. 1–4

In [ ]:
d01 = distance(df.loc[0, PIXELS].tolist(), df.loc[1, PIXELS].tolist())
X_tr, X_te, y_tr, y_te = train_test_split(df[PIXELS], df['label'], test_size=0.25,
                                          random_state=1, stratify=df['label'])
acc_5 = float(accuracy_score(y_te, KNeighborsClassifier(5).fit(X_tr, y_tr).predict(X_te)))
broken = df[PIXELS].copy()
broken['ink_thousands'] = df[PIXELS].sum(axis=1) * 1000
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(broken, df['label'], test_size=0.25,
                                              random_state=0, stratify=df['label'])
acc_broken = float(accuracy_score(yb_te, KNeighborsClassifier(5).fit(Xb_tr, yb_tr).predict(Xb_te)))
fixed = scale_min_max(broken)
Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(fixed, df['label'], test_size=0.25,
                                              random_state=0, stratify=df['label'])
acc_fixed = float(accuracy_score(yf_te, KNeighborsClassifier(5).fit(Xf_tr, yf_tr).predict(Xf_te)))
WHY_SCALE = (
    'Расстояние складывает квадраты разностей по всем столбцам. Столбец с числами в тысячах '
    'даёт вклад в миллионы, а пиксели 0..16 — единицы: сосед выбирается только по этому столбцу. '
    'После min-max все столбцы лежат в [0, 1] и участвуют в сравнении сопоставимо.'
)
print(round(d01, 2), round(acc_5, 4), round(acc_broken, 4), round(acc_fixed, 4))